In [1]:
# Mount Drive (nếu bạn lưu data/code trên Drive)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls -lh /content/drive/MyDrive/UrbanSound8K.zip


-rw------- 1 root root 12G Apr 12 20:04 /content/drive/MyDrive/UrbanSound8K.zip


In [3]:
!unzip -q "/content/drive/MyDrive/UrbanSound8K.zip" -d /content/

In [4]:
%cd /content/UrbanSound8K

/content/UrbanSound8K


In [ ]:
%cd /content/UrbanSound8K
!pip install -r requirements.txt

/content/UrbanSound8K
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 122.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 152.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.0/107.0 kB 20.6 MB/s eta 0:00:00
  Attempting uninstall: parso
    Found existing installation: parso 0.8.5
    Uninstalling parso-0.8.5:
      Successfully uninstalled parso-0.8.5


In [5]:
%%writefile src/cache_features.py
from pathlib import Path
import argparse
import time

import pandas as pd
import torch
from tqdm import tqdm

from config import (
    METADATA_PATH,
    AUDIO_DIR,
    CLASS_NAMES,
    OUTPUT_DIR,
)
from preprocess import preprocess_audio_file


CACHE_DIR = OUTPUT_DIR / "cache"
CACHE_PATH = CACHE_DIR / "features.pt"

CLASS_TO_INDEX = {name: idx for idx, name in enumerate(CLASS_NAMES)}


def build_cache(cache_path=CACHE_PATH, force=False, dtype=torch.float16):
    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    if cache_path.exists() and not force:
        print(f"Cache đã tồn tại: {cache_path}")
        print("Dùng --force để build lại.")
        return cache_path

    df = pd.read_csv(METADATA_PATH)
    df["audio_path"] = df.apply(
        lambda row: AUDIO_DIR / f"fold{int(row['fold'])}" / row["slice_file_name"],
        axis=1,
    )
    df = df[df["audio_path"].apply(lambda p: p.exists())].reset_index(drop=True)

    print(f"Sẽ cache {len(df)} samples vào {cache_path}")

    features_list = []
    labels = []
    folds = []
    file_names = []
    class_names = []

    start = time.time()
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Caching log-Mel"):
        feature = preprocess_audio_file(row["audio_path"])
        features_list.append(feature.to(dtype))
        labels.append(CLASS_TO_INDEX[row["class"]])
        folds.append(int(row["fold"]))
        file_names.append(row["slice_file_name"])
        class_names.append(row["class"])

    features = torch.stack(features_list)
    labels_tensor = torch.tensor(labels, dtype=torch.long)
    folds_tensor = torch.tensor(folds, dtype=torch.long)

    payload = {
        "features": features,
        "labels": labels_tensor,
        "folds": folds_tensor,
        "file_names": file_names,
        "class_names": class_names,
    }

    torch.save(payload, cache_path)

    elapsed = time.time() - start
    size_mb = cache_path.stat().st_size / (1024 ** 2)
    print()
    print(f"Done in {elapsed:.1f}s")
    print(f"Saved to {cache_path} ({size_mb:.1f} MB)")
    print(f"Feature tensor: shape={tuple(features.shape)}, dtype={features.dtype}")

    return cache_path


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--force", action="store_true", help="Build lại dù cache đã tồn tại")
    args = parser.parse_args()

    build_cache(force=args.force)


if __name__ == "__main__":
    main()


Writing src/cache_features.py


In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
NVIDIA A100-SXM4-80GB


In [7]:
!pip install wandb --upgrade
import wandb
wandb.login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 122.1 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.24.0
    Uninstalling wandb-0.24.0:
      Successfully uninstalled wandb-0.24.0


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tien-ngozack2004 (tien-ngozack2004-ho-chi-minh-city-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
%cd /content/UrbanSound8K/src
!python cache_features.py

/content/UrbanSound8K/src
Sẽ cache 8732 samples vào /content/UrbanSound8K/outputs/cache/features.pt
Caching log-Mel: 100% 8732/8732 [00:57<00:00, 151.83it/s]

Done in 58.2s
Saved to /content/UrbanSound8K/outputs/cache/features.pt (369.2 MB)
Feature tensor: shape=(8732, 1, 128, 173), dtype=torch.float16


In [9]:
%%writefile train.py
from pathlib import Path
import csv
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import wandb

from config import CLASS_NAMES, BATCH_SIZE
from dataset import UrbanSoundDataLoader
from cnn_baseline import CNNBaseline


NUM_EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42

SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 2
MIN_LR = 1e-6

EARLY_STOPPING_PATIENCE = 5

WANDB_PROJECT = "urban-sound-classification"
WANDB_RUN_NAME = "cnn_baseline"
WANDB_GROUP = "cnn_baseline"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch in loader:
        features = batch["feature"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * features.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            features = batch["feature"].to(device)
            labels = batch["label"].to(device)

            outputs = model(features)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * features.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


def save_history(history, save_path):
    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "lr", "train_loss", "train_acc", "val_loss", "val_acc"])
        for row in history:
            writer.writerow([
                row["epoch"],
                row["lr"],
                row["train_loss"],
                row["train_acc"],
                row["val_loss"],
                row["val_acc"],
            ])


def main():
    set_seed(SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    output_dir = Path(__file__).resolve().parent.parent / "outputs"
    checkpoint_dir = output_dir / "checkpoints"
    log_dir = output_dir / "logs"

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    data_module = UrbanSoundDataLoader()
    train_loader, val_loader, test_loader = data_module.get_dataloaders()

    model = CNNBaseline(num_classes=len(CLASS_NAMES)).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=MIN_LR,
    )

    run = wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        group=WANDB_GROUP,
        job_type="train",
        config={
            "model_name": "cnn_baseline",
            "num_classes": len(CLASS_NAMES),
            "batch_size": BATCH_SIZE,
            "num_epochs": NUM_EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "seed": SEED,
            "scheduler_factor": SCHEDULER_FACTOR,
            "scheduler_patience": SCHEDULER_PATIENCE,
            "min_lr": MIN_LR,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        },
    )

    best_val_acc = 0.0
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    print("Device:", device)
    print("Train size:", len(data_module.train_dataset))
    print("Val size:", len(data_module.val_dataset))
    print("Test size:", len(data_module.test_dataset))

    for epoch in range(NUM_EPOCHS):
        current_lr = optimizer.param_groups[0]["lr"]

        train_loss, train_acc = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
        )

        val_loss, val_acc = validate_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        scheduler.step(val_loss)

        history.append({
            "epoch": epoch + 1,
            "lr": current_lr,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })

        run.log({
            "epoch": epoch + 1,
            "lr": current_lr,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "best_val_acc_so_far": max(best_val_acc, val_acc),
        })

        print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}]")
        print(f"LR:         {current_lr:.6f}")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            epochs_without_improvement = 0
            best_model_path = checkpoint_dir / "cnn_baseline_best.pth"
            torch.save(model.state_dict(), best_model_path)
            run.summary["best_val_acc"] = best_val_acc
            run.summary["best_epoch"] = best_epoch
            run.summary["best_model_path"] = str(best_model_path)
            print("Best model saved.")
        else:
            epochs_without_improvement += 1
            print(f"No improvement count: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")

        print("-" * 50)

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

    last_model_path = checkpoint_dir / "cnn_baseline_last.pth"
    history_path = log_dir / "cnn_baseline_history.csv"

    torch.save(model.state_dict(), last_model_path)
    save_history(history, history_path)

    run.summary["final_epoch"] = history[-1]["epoch"]
    run.summary["last_model_path"] = str(last_model_path)
    run.summary["history_path"] = str(history_path)
    run.finish()

    print("Training finished.")
    print(f"Best Val Acc: {best_val_acc:.4f}")
    print(f"Best Epoch: {best_epoch}")
    print(f"Best model path: {checkpoint_dir / 'cnn_baseline_best.pth'}")
    print(f"Last model path: {last_model_path}")
    print(f"History path: {history_path}")


if __name__ == "__main__":
    main()

Overwriting train.py


In [11]:
!python train.py

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: tien-ngozack2004 (tien-ngozack2004-ho-chi-minh-city-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run ucp1i6dn (0.5s)
wandb: ⣯ setting up run ucp1i6dn (0.5s)
wandb: ⣟ setting up run ucp1i6dn (0.5s)
wandb: ⡿ setting up run ucp1i6dn (0.5s)
wandb: ⢿ setting up run ucp1i6dn (0.5s)
wandb: ⣻ setting up run ucp1i6dn (1.0s)
wandb: ⣽ setting up run ucp1i6dn (1.0s)
wandb: ⣾ setting up run ucp1i6dn (1.0s)
wandb: ⣷ setting up run ucp1i6dn (1.0s)
wandb: ⣯ setting up run ucp1i6dn (1.0s)
wandb: ⣟ setting up run ucp1i6dn (1.5s)
wandb: ⡿ setting up run ucp1i6dn (1.5s)
wandb: ⢿ setting up run ucp1i6dn (1.5s)
wandb: ⣻ setting up run ucp1i6dn (1.5s)
wandb: ⣽ setting up run ucp

In [12]:
%%writefile evaluate.py
from pathlib import Path
import csv
import json

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import wandb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from config import CLASS_NAMES
from dataset import UrbanSoundDataLoader
from cnn_baseline import CNNBaseline


WANDB_PROJECT = "urban-sound-classification"
WANDB_RUN_NAME = "cnn_baseline_eval"
WANDB_GROUP = "cnn_baseline"


def evaluate_model(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    total = 0

    all_labels = []
    all_preds = []
    all_probs = []
    all_file_names = []
    all_class_names = []

    with torch.no_grad():
        for batch in loader:
            features = batch["feature"].to(device)
            labels = batch["label"].to(device)

            outputs = model(features)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)

            running_loss += loss.item() * features.size(0)
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
            all_file_names.extend(batch["file_name"])
            all_class_names.extend(batch["class_name"])

    test_loss = running_loss / total
    accuracy = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    cm = confusion_matrix(all_labels, all_preds)
    report_dict = classification_report(
        all_labels,
        all_preds,
        target_names=CLASS_NAMES,
        output_dict=True,
        digits=4,
        zero_division=0,
    )
    report_text = classification_report(
        all_labels,
        all_preds,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )

    return {
        "test_loss": test_loss,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "confusion_matrix": cm,
        "report_dict": report_dict,
        "report_text": report_text,
        "labels": all_labels,
        "preds": all_preds,
        "probs": all_probs,
        "file_names": all_file_names,
        "class_names": all_class_names,
    }


def save_metrics(metrics, save_path):
    data = {
        "test_loss": metrics["test_loss"],
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
    }

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)


def save_classification_report(report_dict, save_path):
    rows = []

    for label, values in report_dict.items():
        if isinstance(values, dict):
            row = {"label": label}
            row.update(values)
            rows.append(row)

    fieldnames = sorted({key for row in rows for key in row.keys()})

    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def save_predictions(labels, preds, file_names, class_names, save_path):
    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["file_name", "true_class_name", "true_label", "pred_label", "pred_class_name"])

        for file_name, true_class_name, true_label, pred_label in zip(file_names, class_names, labels, preds):
            writer.writerow([
                file_name,
                true_class_name,
                true_label,
                pred_label,
                CLASS_NAMES[pred_label],
            ])


def save_confusion_matrix_csv(cm, save_path):
    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["true/pred"] + CLASS_NAMES)

        for class_name, row in zip(CLASS_NAMES, cm):
            writer.writerow([class_name] + row.tolist())


def plot_confusion_matrix(cm, class_names, save_path, title):
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm)

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.set_title(title)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")

    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    root_dir = Path(__file__).resolve().parent.parent
    checkpoint_path = root_dir / "outputs" / "checkpoints" / "cnn_baseline_best.pth"
    evaluation_dir = root_dir / "outputs" / "evaluation"
    evaluation_dir.mkdir(parents=True, exist_ok=True)

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Không tìm thấy checkpoint: {checkpoint_path}")

    data_module = UrbanSoundDataLoader()
    train_loader, val_loader, test_loader = data_module.get_dataloaders()

    model = CNNBaseline(num_classes=len(CLASS_NAMES)).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    criterion = nn.CrossEntropyLoss()

    metrics = evaluate_model(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device,
    )

    metrics_path = evaluation_dir / "cnn_baseline_test_metrics.json"
    report_path = evaluation_dir / "cnn_baseline_classification_report.csv"
    predictions_path = evaluation_dir / "cnn_baseline_test_predictions.csv"
    cm_csv_path = evaluation_dir / "cnn_baseline_confusion_matrix.csv"
    cm_img_path = evaluation_dir / "cnn_baseline_confusion_matrix.png"

    save_metrics(metrics, metrics_path)
    save_classification_report(metrics["report_dict"], report_path)
    save_predictions(
        metrics["labels"],
        metrics["preds"],
        metrics["file_names"],
        metrics["class_names"],
        predictions_path,
    )
    save_confusion_matrix_csv(metrics["confusion_matrix"], cm_csv_path)
    plot_confusion_matrix(
        metrics["confusion_matrix"],
        CLASS_NAMES,
        cm_img_path,
        "CNN Baseline Confusion Matrix",
    )

    run = wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        group=WANDB_GROUP,
        job_type="evaluate",
        config={
            "model_name": "cnn_baseline",
            "checkpoint_path": str(checkpoint_path),
            "num_classes": len(CLASS_NAMES),
        },
    )

    run.log({
        "test_loss": metrics["test_loss"],
        "test_accuracy": metrics["accuracy"],
        "test_macro_f1": metrics["macro_f1"],
        "confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=metrics["labels"],
            preds=metrics["preds"],
            class_names=CLASS_NAMES,
        ),
        "confusion_matrix_image": wandb.Image(str(cm_img_path)),
    })

    run.summary["metrics_path"] = str(metrics_path)
    run.summary["report_path"] = str(report_path)
    run.summary["predictions_path"] = str(predictions_path)
    run.summary["confusion_matrix_csv_path"] = str(cm_csv_path)
    run.summary["confusion_matrix_image_path"] = str(cm_img_path)
    run.finish()

    print("Evaluation finished.")
    print(f"Checkpoint: {checkpoint_path}")
    print(f"Test Loss: {metrics['test_loss']:.4f}")
    print(f"Test Accuracy: {metrics['accuracy']:.4f}")
    print(f"Test Macro-F1: {metrics['macro_f1']:.4f}")
    print()
    print(metrics["report_text"])
    print(f"Saved metrics to: {metrics_path}")
    print(f"Saved report to: {report_path}")
    print(f"Saved predictions to: {predictions_path}")
    print(f"Saved confusion matrix csv to: {cm_csv_path}")
    print(f"Saved confusion matrix image to: {cm_img_path}")


if __name__ == "__main__":
    main()

Overwriting evaluate.py


In [13]:
!python evaluate.py

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: tien-ngozack2004 (tien-ngozack2004-ho-chi-minh-city-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run ktud0kjg (0.5s)
wandb: ⣯ setting up run ktud0kjg (0.5s)
wandb: ⣟ setting up run ktud0kjg (0.5s)
wandb: ⡿ setting up run ktud0kjg (0.5s)
wandb: ⢿ setting up run ktud0kjg (0.5s)
wandb: ⣻ setting up run ktud0kjg (1.0s)
wandb: ⣽ setting up run ktud0kjg (1.0s)
wandb: ⣾ setting up run ktud0kjg (1.0s)
wandb: ⣷ setting up run ktud0kjg (1.0s)
wandb: ⣯ setting up run ktud0kjg (1.0s)
wandb: ⣟ setting up run ktud0kjg (1.5s)
wandb: ⡿ setting up run ktud0kjg (1.5s)
wandb: ⢿ setting up run ktud0kjg (1.5s)
wandb: ⣻ setting up run ktud0kjg (1.5s)
wandb: ⣽ setting up run ktu

RESNET

In [14]:
%%writefile resnet.py
import torch
import torch.nn as nn

def conv3x3(in_channels, out_channels, stride=1):
    return nn.Conv2d(
        in_channels,
        out_channels,
        kernel_size=3,
        stride=stride,
        padding=1,
        bias=False,
    )


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = conv3x3(in_channels, out_channels, stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout2d(dropout)
        self.conv2 = conv3x3(out_channels, out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out


class ResNet18(nn.Module):
    def __init__(self, num_classes=10, base_channels=32, dropout=0.5):
        super().__init__()

        self.in_channels = base_channels

        self.stem = nn.Sequential(
            nn.Conv2d(1, base_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
        )

        self.layer1 = self._make_layer(base_channels, blocks=2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(base_channels * 2, blocks=2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(base_channels * 4, blocks=2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(base_channels * 8, blocks=2, stride=2, dropout=dropout)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(base_channels * 8, num_classes)
        )

    def _make_layer(self, out_channels, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

        layers = [
            BasicBlock(
                self.in_channels,
                out_channels,
                stride=stride,
                downsample=downsample,
                dropout=dropout,
            )
        ]
        self.in_channels = out_channels

        for _ in range(1, blocks):
            layers.append(
                BasicBlock(
                    self.in_channels,
                    out_channels,
                    stride=1,
                    downsample=None,
                    dropout=dropout,
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


if __name__ == "__main__":
    model = ResNet18(num_classes=10, base_channels=32, dropout=0.5)
    x = torch.randn(16, 1, 128, 173)
    y = model(x)

    print("Input shape:", x.shape)
    print("Output shape:", y.shape)

Writing resnet.py


In [15]:
!python resnet.py

Input shape: torch.Size([16, 1, 128, 173])
Output shape: torch.Size([16, 10])


In [ ]:
!git pull
!grep "MIXUP_PROB" train_resnet.py
!grep "MIXUP_PROB" train_resnet_aug.py
!grep "BASE_CHANNELS" train_resnet_aug.py


Already up to date.
MIXUP_PROB = 0.0
        use_mixup = random.random() < MIXUP_PROB
            "mixup_prob": MIXUP_PROB,
                    "mixup_prob": MIXUP_PROB,
            "mixup_prob": MIXUP_PROB,
MIXUP_PROB = 0.0
        use_mixup = random.random() < MIXUP_PROB
            "mixup_prob": MIXUP_PROB,
BASE_CHANNELS = 32
        base_channels=BASE_CHANNELS,
            "base_channels": BASE_CHANNELS,


In [16]:
%%writefile train_resnet.py
from pathlib import Path
import csv
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import wandb
from sklearn.metrics import f1_score

from config import CLASS_NAMES, BATCH_SIZE
from dataset import UrbanSoundDataLoader
from resnet import ResNet18


NUM_EPOCHS = 40
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
SEED = 42

EARLY_STOPPING_PATIENCE = 8
LABEL_SMOOTHING = 0.05
DROPOUT = 0.3
BASE_CHANNELS = 48
GRAD_CLIP = 3.0

MIXUP_ALPHA = 0.2
MIXUP_PROB = 0.0

RUN_TAG = "resnet18_regular_v3"
WANDB_PROJECT = "urban-sound-classification"
WANDB_RUN_NAME = RUN_TAG
WANDB_GROUP = "resnet18_regular"


def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def mixup_data(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, y, 1.0

    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a = y
    y_b = y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for batch in loader:
        features = batch["feature"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad(set_to_none=True)

        use_mixup = random.random() < MIXUP_PROB

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            if use_mixup:
                mixed_features, labels_a, labels_b, lam = mixup_data(features, labels, MIXUP_ALPHA)
                outputs = model(mixed_features)
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            else:
                outputs = model(features)
                loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * features.size(0)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = float((np.array(all_preds) == np.array(all_labels)).mean())
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1


def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            features = batch["feature"].to(device)
            labels = batch["label"].to(device)

            outputs = model(features)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * features.size(0)
            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = float((np.array(all_preds) == np.array(all_labels)).mean())
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1


def save_history(history, save_path):
    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch",
            "lr",
            "train_loss",
            "train_acc",
            "train_macro_f1",
            "val_loss",
            "val_acc",
            "val_macro_f1",
        ])
        for row in history:
            writer.writerow([
                row["epoch"],
                row["lr"],
                row["train_loss"],
                row["train_acc"],
                row["train_macro_f1"],
                row["val_loss"],
                row["val_acc"],
                row["val_macro_f1"],
            ])


def main():
    set_seed(SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    output_dir = Path(__file__).resolve().parent.parent / "outputs"
    checkpoint_dir = output_dir / "checkpoints"
    log_dir = output_dir / "logs"

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    data_module = UrbanSoundDataLoader(batch_size=BATCH_SIZE, num_workers=0)
    train_loader, val_loader, test_loader = data_module.get_dataloaders()

    model = ResNet18(
        num_classes=len(CLASS_NAMES),
        base_channels=BASE_CHANNELS,
        dropout=DROPOUT,
    ).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=NUM_EPOCHS,
        eta_min=1e-6,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_model_path = checkpoint_dir / f"{RUN_TAG}_best.pth"
    last_model_path = checkpoint_dir / f"{RUN_TAG}_last.pth"
    history_path = log_dir / f"{RUN_TAG}_history.csv"

    run = wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        group=WANDB_GROUP,
        job_type="train",
        config={
            "run_tag": RUN_TAG,
            "model_name": "resnet18_regular",
            "num_classes": len(CLASS_NAMES),
            "batch_size": BATCH_SIZE,
            "num_epochs": NUM_EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "seed": SEED,
            "dropout": DROPOUT,
            "base_channels": BASE_CHANNELS,
            "label_smoothing": LABEL_SMOOTHING,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "grad_clip": GRAD_CLIP,
            "mixup_alpha": MIXUP_ALPHA,
            "mixup_prob": MIXUP_PROB,
        },
    )

    best_val_acc = 0.0
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    print("Device:", device)
    print("Run tag:", RUN_TAG)
    print("Train size:", len(data_module.train_dataset))
    print("Val size:", len(data_module.val_dataset))
    print("Test size:", len(data_module.test_dataset))

    for epoch in range(NUM_EPOCHS):
        current_lr = optimizer.param_groups[0]["lr"]

        train_loss, train_acc, train_f1 = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
        )

        val_loss, val_acc, val_f1 = validate_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        scheduler.step()

        history.append({
            "epoch": epoch + 1,
            "lr": current_lr,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_macro_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_macro_f1": val_f1,
        })

        run.log({
            "epoch": epoch + 1,
            "lr": current_lr,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_macro_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_macro_f1": val_f1,
            "best_val_acc_so_far": max(best_val_acc, val_acc),
        })

        print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}]")
        print(f"LR:         {current_lr:.6f}")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val F1:   {val_f1:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            epochs_without_improvement = 0
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "run_tag": RUN_TAG,
                    "base_channels": BASE_CHANNELS,
                    "dropout": DROPOUT,
                    "best_val_acc": best_val_acc,
                    "best_epoch": best_epoch,
                    "seed": SEED,
                    "mixup_alpha": MIXUP_ALPHA,
                    "mixup_prob": MIXUP_PROB,
                },
                best_model_path,
            )
            run.summary["best_val_acc"] = best_val_acc
            run.summary["best_epoch"] = best_epoch
            run.summary["best_model_path"] = str(best_model_path)
            print("Best model saved.")
        else:
            epochs_without_improvement += 1
            print(f"No improvement count: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")

        print("-" * 50)

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "run_tag": RUN_TAG,
            "base_channels": BASE_CHANNELS,
            "dropout": DROPOUT,
            "seed": SEED,
            "mixup_alpha": MIXUP_ALPHA,
            "mixup_prob": MIXUP_PROB,
        },
        last_model_path,
    )
    save_history(history, history_path)

    run.summary["final_epoch"] = history[-1]["epoch"]
    run.summary["last_model_path"] = str(last_model_path)
    run.summary["history_path"] = str(history_path)
    run.finish()

    print("Training finished.")
    print(f"Best Val Acc: {best_val_acc:.4f}")
    print(f"Best Epoch: {best_epoch}")
    print(f"Best model path: {best_model_path}")
    print(f"Last model path: {last_model_path}")
    print(f"History path: {history_path}")


if __name__ == "__main__":
    main()

Writing train_resnet.py


In [17]:
!python train_resnet.py

/content/UrbanSound8K/src/train_resnet.py:198: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: tien-ngozack2004 (tien-ngozack2004-ho-chi-minh-city-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run lgqcwhxv (0.5s)
wandb: ⣯ setting up run lgqcwhxv (0.5s)
wandb: ⣟ setting up run lgqcwhxv (0.5s)
wandb: ⡿ setting up run lgqcwhxv (0.5s)
wandb: ⢿ setting up run lgqcwhxv (0.5s)
wandb: ⣻ setting up run lgqcwhxv (1.0s)
wandb: ⣽ setting up run lgqcwhxv (1.0s)
wandb: ⣾ setting up run lgqcwhxv (1.0s)
wandb: ⣷ setting up r

In [18]:
%%writefile evaluate_resnet.py
from pathlib import Path
import csv
import json

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import wandb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from config import CLASS_NAMES
from dataset import UrbanSoundDataLoader
from resnet import ResNet18


RUN_TAG = "resnet18_regular_v3"
WANDB_PROJECT = "urban-sound-classification"
WANDB_RUN_NAME = f"{RUN_TAG}_tta_eval"
WANDB_GROUP = "resnet18_regular"

DEFAULT_BASE_CHANNELS = 32
DEFAULT_DROPOUT = 0.3
TTA_SHIFT = 4


def forward_tta(model, features):
    outputs_1 = model(features)
    outputs_2 = model(torch.roll(features, shifts=TTA_SHIFT, dims=-1))
    outputs_3 = model(torch.roll(features, shifts=-TTA_SHIFT, dims=-1))
    return (outputs_1 + outputs_2 + outputs_3) / 3.0


def evaluate_model(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    total = 0

    all_labels = []
    all_preds = []
    all_probs = []
    all_file_names = []
    all_true_class_names = []

    with torch.no_grad():
        for batch in loader:
            features = batch["feature"].to(device)
            labels = batch["label"].to(device)

            outputs = forward_tta(model, features)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)

            running_loss += loss.item() * features.size(0)
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
            all_file_names.extend(batch["file_name"])
            all_true_class_names.extend(batch["class_name"])

    test_loss = running_loss / total
    accuracy = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    cm = confusion_matrix(all_labels, all_preds)

    report_dict = classification_report(
        all_labels,
        all_preds,
        target_names=CLASS_NAMES,
        output_dict=True,
        digits=4,
        zero_division=0,
    )
    report_text = classification_report(
        all_labels,
        all_preds,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )

    return {
        "test_loss": test_loss,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "confusion_matrix": cm,
        "report_dict": report_dict,
        "report_text": report_text,
        "labels": all_labels,
        "preds": all_preds,
        "probs": all_probs,
        "file_names": all_file_names,
        "true_class_names": all_true_class_names,
    }


def save_metrics(metrics, save_path):
    data = {
        "test_loss": metrics["test_loss"],
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
    }

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)


def save_classification_report(report_dict, save_path):
    rows = []

    for label, values in report_dict.items():
        if isinstance(values, dict):
            row = {"label": label}
            row.update(values)
            rows.append(row)

    fieldnames = sorted({key for row in rows for key in row.keys()})

    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def save_predictions(labels, preds, probs, file_names, true_class_names, save_path):
    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "file_name",
            "true_class_name",
            "true_label",
            "pred_label",
            "pred_class_name",
            "pred_confidence",
        ])

        for file_name, true_class_name, true_label, pred_label, prob_vector in zip(
            file_names, true_class_names, labels, preds, probs
        ):
            pred_confidence = float(prob_vector[pred_label])
            writer.writerow([
                file_name,
                true_class_name,
                true_label,
                pred_label,
                CLASS_NAMES[pred_label],
                pred_confidence,
            ])


def save_confusion_matrix_csv(cm, save_path):
    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["true/pred"] + CLASS_NAMES)

        for class_name, row in zip(CLASS_NAMES, cm):
            writer.writerow([class_name] + row.tolist())


def plot_confusion_matrix(cm, class_names, save_path, title):
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm)

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.set_title(title)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")

    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    root_dir = Path(__file__).resolve().parent.parent
    checkpoint_path = root_dir / "outputs" / "checkpoints" / f"{RUN_TAG}_best.pth"
    evaluation_dir = root_dir / "outputs" / "evaluation"
    evaluation_dir.mkdir(parents=True, exist_ok=True)

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Không tìm thấy checkpoint: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
        base_channels = checkpoint.get("base_channels", DEFAULT_BASE_CHANNELS)
        dropout = checkpoint.get("dropout", DEFAULT_DROPOUT)
    else:
        state_dict = checkpoint
        base_channels = DEFAULT_BASE_CHANNELS
        dropout = DEFAULT_DROPOUT

    data_module = UrbanSoundDataLoader(batch_size=None if False else None)
    data_module = UrbanSoundDataLoader(num_workers=0)
    _, _, test_loader = data_module.get_dataloaders()

    model = ResNet18(
        num_classes=len(CLASS_NAMES),
        base_channels=base_channels,
        dropout=dropout,
    ).to(device)

    model.load_state_dict(state_dict)

    criterion = nn.CrossEntropyLoss()

    metrics = evaluate_model(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device,
    )

    metrics_path = evaluation_dir / f"{RUN_TAG}_test_metrics_tta.json"
    report_path = evaluation_dir / f"{RUN_TAG}_classification_report_tta.csv"
    predictions_path = evaluation_dir / f"{RUN_TAG}_test_predictions_tta.csv"
    cm_csv_path = evaluation_dir / f"{RUN_TAG}_confusion_matrix_tta.csv"
    cm_img_path = evaluation_dir / f"{RUN_TAG}_confusion_matrix_tta.png"

    save_metrics(metrics, metrics_path)
    save_classification_report(metrics["report_dict"], report_path)
    save_predictions(
        metrics["labels"],
        metrics["preds"],
        metrics["probs"],
        metrics["file_names"],
        metrics["true_class_names"],
        predictions_path,
    )
    save_confusion_matrix_csv(metrics["confusion_matrix"], cm_csv_path)
    plot_confusion_matrix(
        metrics["confusion_matrix"],
        CLASS_NAMES,
        cm_img_path,
        f"{RUN_TAG} TTA Confusion Matrix",
    )

    run = wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        group=WANDB_GROUP,
        job_type="evaluate",
        config={
            "run_tag": RUN_TAG,
            "checkpoint_path": str(checkpoint_path),
            "num_classes": len(CLASS_NAMES),
            "base_channels": base_channels,
            "dropout": dropout,
            "tta_shift": TTA_SHIFT,
        },
    )

    run.log({
        "test_loss": metrics["test_loss"],
        "test_accuracy": metrics["accuracy"],
        "test_macro_f1": metrics["macro_f1"],
        "confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=metrics["labels"],
            preds=metrics["preds"],
            class_names=CLASS_NAMES,
        ),
        "confusion_matrix_image": wandb.Image(str(cm_img_path)),
    })

    run.summary["metrics_path"] = str(metrics_path)
    run.summary["report_path"] = str(report_path)
    run.summary["predictions_path"] = str(predictions_path)
    run.summary["confusion_matrix_csv_path"] = str(cm_csv_path)
    run.summary["confusion_matrix_image_path"] = str(cm_img_path)
    run.finish()

    print("Evaluation finished.")
    print(f"Checkpoint: {checkpoint_path}")
    print(f"Test Loss: {metrics['test_loss']:.4f}")
    print(f"Test Accuracy: {metrics['accuracy']:.4f}")
    print(f"Test Macro-F1: {metrics['macro_f1']:.4f}")
    print()
    print(metrics["report_text"])
    print(f"Saved metrics to: {metrics_path}")
    print(f"Saved report to: {report_path}")
    print(f"Saved predictions to: {predictions_path}")
    print(f"Saved confusion matrix csv to: {cm_csv_path}")
    print(f"Saved confusion matrix image to: {cm_img_path}")


if __name__ == "__main__":
    main()

Writing evaluate_resnet.py


In [19]:
!python evaluate_resnet.py

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: tien-ngozack2004 (tien-ngozack2004-ho-chi-minh-city-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run elzz92rq (0.5s)
wandb: ⣯ setting up run elzz92rq (0.5s)
wandb: ⣟ setting up run elzz92rq (0.5s)
wandb: ⡿ setting up run elzz92rq (0.5s)
wandb: ⢿ setting up run elzz92rq (0.5s)
wandb: ⣻ setting up run elzz92rq (1.0s)
wandb: ⣽ setting up run elzz92rq (1.0s)
wandb: ⣾ setting up run elzz92rq (1.0s)
wandb: ⣷ setting up run elzz92rq (1.0s)
wandb: ⣯ setting up run elzz92rq (1.0s)
wandb: ⣟ setting up run elzz92rq (1.5s)
wandb: ⡿ setting up run elzz92rq (1.5s)
wandb: ⢿ setting up run elzz92rq (1.5s)
wandb: ⣻ setting up run elzz92rq (1.5s)
wandb: ⣽ setting up run elz

In [24]:
%%writefile dataset_aug.py
import random

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

from config import (
    METADATA_PATH,
    AUDIO_DIR,
    CLASS_NAMES,
    BATCH_SIZE,
    NUM_WORKERS,
    TEST_FOLD,
    VAL_FOLD,
    OUTPUT_DIR,
)
from preprocess import preprocess_audio_file


CLASS_TO_INDEX = {class_name: idx for idx, class_name in enumerate(CLASS_NAMES)}
INDEX_TO_CLASS = {idx: class_name for class_name, idx in CLASS_TO_INDEX.items()}

CACHE_PATH = OUTPUT_DIR / "cache" / "features.pt"

_CACHE_SINGLETON = None


def _load_cache():
    global _CACHE_SINGLETON
    if _CACHE_SINGLETON is None:
        _CACHE_SINGLETON = torch.load(CACHE_PATH, map_location="cpu", weights_only=False)
    return _CACHE_SINGLETON


class SpectrogramAugmentation:
    def __init__(
        self,
        noise_prob=0.3,
        noise_std=0.01,
        time_shift_prob=0.5,
        max_time_shift=12,
        freq_mask_prob=0.4,
        max_freq_mask_width=16,
        num_freq_masks=1,
        time_mask_prob=0.4,
        max_time_mask_width=15,
        num_time_masks=1,
    ):
        self.noise_prob = noise_prob
        self.noise_std = noise_std
        self.time_shift_prob = time_shift_prob
        self.max_time_shift = max_time_shift
        self.freq_mask_prob = freq_mask_prob
        self.max_freq_mask_width = max_freq_mask_width
        self.num_freq_masks = num_freq_masks
        self.time_mask_prob = time_mask_prob
        self.max_time_mask_width = max_time_mask_width
        self.num_time_masks = num_time_masks

    def add_noise(self, x):
        noise = torch.randn_like(x) * self.noise_std
        return x + noise

    def time_shift(self, x):
        max_shift = min(self.max_time_shift, x.size(2) - 1)
        if max_shift <= 0:
            return x
        shift = random.randint(-max_shift, max_shift)
        return torch.roll(x, shifts=shift, dims=2)

    def freq_mask(self, x):
        max_width = min(self.max_freq_mask_width, x.size(1))
        if max_width <= 0:
            return x

        for _ in range(self.num_freq_masks):
            width = random.randint(1, max_width)
            if x.size(1) - width < 0:
                continue
            start = random.randint(0, x.size(1) - width)
            x[:, start:start + width, :] = 0.0
        return x

    def time_mask(self, x):
        max_width = min(self.max_time_mask_width, x.size(2))
        if max_width <= 0:
            return x

        for _ in range(self.num_time_masks):
            width = random.randint(1, max_width)
            if x.size(2) - width < 0:
                continue
            start = random.randint(0, x.size(2) - width)
            x[:, :, start:start + width] = 0.0
        return x

    def __call__(self, x):
        x = x.clone()

        if random.random() < self.noise_prob:
            x = self.add_noise(x)

        if random.random() < self.time_shift_prob:
            x = self.time_shift(x)

        if random.random() < self.freq_mask_prob:
            x = self.freq_mask(x)

        if random.random() < self.time_mask_prob:
            x = self.time_mask(x)

        return x


class UrbanSoundDatasetAug(Dataset):
    def __init__(
        self,
        split,
        test_fold,
        val_fold=None,
        metadata_path=METADATA_PATH,
        audio_dir=AUDIO_DIR,
        augment=False,
        augmenter=None,
    ):
        self.split = split
        self.test_fold = int(test_fold)
        self.val_fold = None if val_fold is None else int(val_fold)
        self.metadata_path = metadata_path
        self.audio_dir = audio_dir
        self.augment = augment and split == "train"
        self.augmenter = augmenter if augmenter is not None else SpectrogramAugmentation()

        self.use_cache = CACHE_PATH.exists()

        if self.use_cache:
            self._init_from_cache()
        else:
            self._init_from_audio()

    def _split_mask(self, folds_tensor):
        if self.split == "train":
            if self.val_fold is None:
                raise ValueError("val_fold không được để None khi split='train'")
            return (folds_tensor != self.test_fold) & (folds_tensor != self.val_fold)
        if self.split == "val":
            if self.val_fold is None:
                raise ValueError("val_fold không được để None khi split='val'")
            return folds_tensor == self.val_fold
        if self.split == "test":
            return folds_tensor == self.test_fold
        raise ValueError("split phải là 'train', 'val' hoặc 'test'")

    def _init_from_cache(self):
        cache = _load_cache()
        folds = cache["folds"]
        mask = self._split_mask(folds)
        idx_list = torch.where(mask)[0].tolist()

        self._features = cache["features"][mask].contiguous()
        self._labels = cache["labels"][mask].contiguous()
        self._folds = cache["folds"][mask].contiguous()
        self._file_names = [cache["file_names"][i] for i in idx_list]
        self._class_names = [cache["class_names"][i] for i in idx_list]

    def _init_from_audio(self):
        df = pd.read_csv(self.metadata_path)
        df["audio_path"] = df.apply(
            lambda row: self.audio_dir / f"fold{int(row['fold'])}" / row["slice_file_name"],
            axis=1,
        )
        df = df[df["audio_path"].apply(lambda x: x.exists())].reset_index(drop=True)
        df["label"] = df["class"].map(CLASS_TO_INDEX)

        folds = torch.tensor(df["fold"].astype(int).values, dtype=torch.long)
        mask = self._split_mask(folds).numpy()
        self.df = df[mask].reset_index(drop=True)

    def __len__(self):
        if self.use_cache:
            return self._labels.size(0)
        return len(self.df)

    def __getitem__(self, idx):
        if self.use_cache:
            feature = self._features[idx].to(torch.float32)
            label = int(self._labels[idx])
            class_name = self._class_names[idx]
            file_name = self._file_names[idx]
            fold = int(self._folds[idx])
            audio_path = ""
        else:
            row = self.df.iloc[idx]
            feature = preprocess_audio_file(row["audio_path"])
            label = int(row["label"])
            class_name = row["class"]
            file_name = row["slice_file_name"]
            fold = int(row["fold"])
            audio_path = str(row["audio_path"])

        if self.augment:
            feature = self.augmenter(feature)

        return {
            "feature": feature,
            "label": torch.tensor(label, dtype=torch.long),
            "class_name": class_name,
            "file_name": file_name,
            "fold": fold,
            "audio_path": audio_path,
        }


class UrbanSoundDataLoaderAug:
    def __init__(
        self,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        test_fold=TEST_FOLD,
        val_fold=VAL_FOLD,
        augmenter=None,
    ):
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.test_fold = test_fold
        self.val_fold = val_fold
        self.augmenter = augmenter if augmenter is not None else SpectrogramAugmentation()

        self.train_dataset = UrbanSoundDatasetAug(
            split="train",
            test_fold=self.test_fold,
            val_fold=self.val_fold,
            augment=True,
            augmenter=self.augmenter,
        )
        self.val_dataset = UrbanSoundDatasetAug(
            split="val",
            test_fold=self.test_fold,
            val_fold=self.val_fold,
            augment=False,
            augmenter=self.augmenter,
        )
        self.test_dataset = UrbanSoundDatasetAug(
            split="test",
            test_fold=self.test_fold,
            val_fold=self.val_fold,
            augment=False,
            augmenter=self.augmenter,
        )

    def get_train_loader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
            persistent_workers=self.num_workers > 0,
        )

    def get_val_loader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
            persistent_workers=self.num_workers > 0,
        )

    def get_test_loader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
            persistent_workers=self.num_workers > 0,
        )

    def get_dataloaders(self):
        return self.get_train_loader(), self.get_val_loader(), self.get_test_loader()


if __name__ == "__main__":
    data_module = UrbanSoundDataLoaderAug()

    train_loader, val_loader, test_loader = data_module.get_dataloaders()

    print("Cache mode:", data_module.train_dataset.use_cache)
    print("Train size:", len(data_module.train_dataset))
    print("Val size:", len(data_module.val_dataset))
    print("Test size:", len(data_module.test_dataset))

    batch = next(iter(train_loader))
    print("Batch feature shape:", batch["feature"].shape)
    print("Batch label shape:", batch["label"].shape)
    print("Batch class names:", batch["class_name"][:5])


Overwriting dataset_aug.py


In [25]:
%%writefile train_resnet_aug.py
from pathlib import Path
import csv
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import wandb
from sklearn.metrics import f1_score

from config import CLASS_NAMES, BATCH_SIZE
from dataset_aug import UrbanSoundDataLoaderAug
from resnet import ResNet18


NUM_EPOCHS = 50
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-3
SEED = 42

EARLY_STOPPING_PATIENCE = 10
LABEL_SMOOTHING = 0.05
DROPOUT = 0.5
BASE_CHANNELS = 64
GRAD_CLIP = 3.0

MIXUP_ALPHA = 0.2
MIXUP_PROB = 0.3

WANDB_PROJECT = "urban-sound-classification"
WANDB_RUN_NAME = "resnet18_aug_full_v2"
WANDB_GROUP = "resnet18_aug"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def mixup_data(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, y, 1.0

    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a = y
    y_b = y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for batch in loader:
        features = batch["feature"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad(set_to_none=True)

        use_mixup = random.random() < MIXUP_PROB

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            if use_mixup:
                mixed_features, labels_a, labels_b, lam = mixup_data(features, labels, MIXUP_ALPHA)
                outputs = model(mixed_features)
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            else:
                outputs = model(features)
                loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * features.size(0)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = float((np.array(all_preds) == np.array(all_labels)).mean())
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1


def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            features = batch["feature"].to(device)
            labels = batch["label"].to(device)

            outputs = model(features)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * features.size(0)
            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = float((np.array(all_preds) == np.array(all_labels)).mean())
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1


def save_history(history, save_path):
    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch",
            "lr",
            "train_loss",
            "train_acc",
            "train_macro_f1",
            "val_loss",
            "val_acc",
            "val_macro_f1",
        ])
        for row in history:
            writer.writerow([
                row["epoch"],
                row["lr"],
                row["train_loss"],
                row["train_acc"],
                row["train_macro_f1"],
                row["val_loss"],
                row["val_acc"],
                row["val_macro_f1"],
            ])


def main():
    set_seed(SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    output_dir = Path(__file__).resolve().parent.parent / "outputs"
    checkpoint_dir = output_dir / "checkpoints"
    log_dir = output_dir / "logs"

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    data_module = UrbanSoundDataLoaderAug()
    train_loader, val_loader, test_loader = data_module.get_dataloaders()

    model = ResNet18(
        num_classes=len(CLASS_NAMES),
        base_channels=BASE_CHANNELS,
        dropout=DROPOUT,
    ).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=NUM_EPOCHS,
        eta_min=1e-6,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    run = wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        group=WANDB_GROUP,
        job_type="train",
        config={
            "model_name": "resnet18_small_mixup",
            "num_classes": len(CLASS_NAMES),
            "batch_size": BATCH_SIZE,
            "num_epochs": NUM_EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "seed": SEED,
            "dropout": DROPOUT,
            "base_channels": BASE_CHANNELS,
            "label_smoothing": LABEL_SMOOTHING,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "grad_clip": GRAD_CLIP,
            "mixup_alpha": MIXUP_ALPHA,
            "mixup_prob": MIXUP_PROB,
        },
    )

    best_val_acc = 0.0
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    print("Device:", device)
    print("Train size:", len(data_module.train_dataset))
    print("Val size:", len(data_module.val_dataset))
    print("Test size:", len(data_module.test_dataset))

    for epoch in range(NUM_EPOCHS):
        current_lr = optimizer.param_groups[0]["lr"]

        train_loss, train_acc, train_f1 = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
        )

        val_loss, val_acc, val_f1 = validate_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        scheduler.step()

        history.append({
            "epoch": epoch + 1,
            "lr": current_lr,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_macro_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_macro_f1": val_f1,
        })

        run.log({
            "epoch": epoch + 1,
            "lr": current_lr,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_macro_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_macro_f1": val_f1,
            "best_val_acc_so_far": max(best_val_acc, val_acc),
        })

        print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}]")
        print(f"LR:         {current_lr:.6f}")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val F1:   {val_f1:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            epochs_without_improvement = 0
            best_model_path = checkpoint_dir / "resnet18_aug_best.pth"
            torch.save(model.state_dict(), best_model_path)
            run.summary["best_val_acc"] = best_val_acc
            run.summary["best_epoch"] = best_epoch
            run.summary["best_model_path"] = str(best_model_path)
            print("Best model saved.")
        else:
            epochs_without_improvement += 1
            print(f"No improvement count: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")

        print("-" * 50)

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

    last_model_path = checkpoint_dir / "resnet18_aug_last.pth"
    history_path = log_dir / "resnet18_aug_history.csv"

    torch.save(model.state_dict(), last_model_path)
    save_history(history, history_path)

    run.summary["final_epoch"] = history[-1]["epoch"]
    run.summary["last_model_path"] = str(last_model_path)
    run.summary["history_path"] = str(history_path)
    run.finish()

    print("Training finished.")
    print(f"Best Val Acc: {best_val_acc:.4f}")
    print(f"Best Epoch: {best_epoch}")
    print(f"Best model path: {checkpoint_dir / 'resnet18_aug_best.pth'}")
    print(f"Last model path: {last_model_path}")
    print(f"History path: {history_path}")


if __name__ == "__main__":
    main()

Overwriting train_resnet_aug.py


In [26]:
!python train_resnet_aug.py

/content/UrbanSound8K/src/train_resnet_aug.py:191: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: tien-ngozack2004 (tien-ngozack2004-ho-chi-minh-city-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 1qv909yv (0.5s)
wandb: ⣯ setting up run 1qv909yv (0.5s)
wandb: ⣟ setting up run 1qv909yv (0.5s)
wandb: ⡿ setting up run 1qv909yv (0.5s)
wandb: ⢿ setting up run 1qv909yv (0.5s)
wandb: ⣻ setting up run 1qv909yv (1.0s)
wandb: ⣽ setting up run 1qv909yv (1.0s)
wandb: ⣾ setting up run 1qv909yv (1.0s)
wandb: ⣷ setting 

In [27]:
%%writefile evaluate_resnet_aug.py
from pathlib import Path
import csv
import json

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import wandb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from config import CLASS_NAMES
from dataset_aug import UrbanSoundDataLoaderAug
from resnet import ResNet18


WANDB_PROJECT = "urban-sound-classification"
WANDB_RUN_NAME = "resnet18_aug_full_tta_eval"
WANDB_GROUP = "resnet18_aug"

DROPOUT = 0.5
BASE_CHANNELS = 64
TTA_SHIFT = 4


def forward_tta(model, features):
    outputs_1 = model(features)
    outputs_2 = model(torch.roll(features, shifts=TTA_SHIFT, dims=-1))
    outputs_3 = model(torch.roll(features, shifts=-TTA_SHIFT, dims=-1))
    return (outputs_1 + outputs_2 + outputs_3) / 3.0


def evaluate_model(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    total = 0

    all_labels = []
    all_preds = []
    all_probs = []
    all_file_names = []
    all_class_names = []

    with torch.no_grad():
        for batch in loader:
            features = batch["feature"].to(device)
            labels = batch["label"].to(device)

            outputs = forward_tta(model, features)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)

            running_loss += loss.item() * features.size(0)
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
            all_file_names.extend(batch["file_name"])
            all_class_names.extend(batch["class_name"])

    test_loss = running_loss / total
    accuracy = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    cm = confusion_matrix(all_labels, all_preds)
    report_dict = classification_report(
        all_labels,
        all_preds,
        target_names=CLASS_NAMES,
        output_dict=True,
        digits=4,
        zero_division=0,
    )
    report_text = classification_report(
        all_labels,
        all_preds,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )

    return {
        "test_loss": test_loss,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "confusion_matrix": cm,
        "report_dict": report_dict,
        "report_text": report_text,
        "labels": all_labels,
        "preds": all_preds,
        "probs": all_probs,
        "file_names": all_file_names,
        "class_names": all_class_names,
    }


def save_metrics(metrics, save_path):
    data = {
        "test_loss": metrics["test_loss"],
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
    }

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)


def save_classification_report(report_dict, save_path):
    rows = []

    for label, values in report_dict.items():
        if isinstance(values, dict):
            row = {"label": label}
            row.update(values)
            rows.append(row)

    fieldnames = sorted({key for row in rows for key in row.keys()})

    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def save_predictions(labels, preds, file_names, class_names, save_path):
    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["file_name", "true_class_name", "true_label", "pred_label", "pred_class_name"])

        for file_name, true_class_name, true_label, pred_label in zip(file_names, class_names, labels, preds):
            writer.writerow([
                file_name,
                true_class_name,
                true_label,
                pred_label,
                CLASS_NAMES[pred_label],
            ])


def save_confusion_matrix_csv(cm, save_path):
    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["true/pred"] + CLASS_NAMES)

        for class_name, row in zip(CLASS_NAMES, cm):
            writer.writerow([class_name] + row.tolist())


def plot_confusion_matrix(cm, class_names, save_path, title):
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm)

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.set_title(title)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")

    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    root_dir = Path(__file__).resolve().parent.parent
    checkpoint_path = root_dir / "outputs" / "checkpoints" / "resnet18_aug_best.pth"
    evaluation_dir = root_dir / "outputs" / "evaluation"
    evaluation_dir.mkdir(parents=True, exist_ok=True)

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Không tìm thấy checkpoint: {checkpoint_path}")

    data_module = UrbanSoundDataLoaderAug()
    train_loader, val_loader, test_loader = data_module.get_dataloaders()

    model = ResNet18(
        num_classes=len(CLASS_NAMES),
        base_channels=BASE_CHANNELS,
        dropout=DROPOUT,
    ).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    criterion = nn.CrossEntropyLoss()

    metrics = evaluate_model(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device,
    )

    metrics_path = evaluation_dir / "resnet18_aug_test_metrics_tta.json"
    report_path = evaluation_dir / "resnet18_aug_classification_report_tta.csv"
    predictions_path = evaluation_dir / "resnet18_aug_test_predictions_tta.csv"
    cm_csv_path = evaluation_dir / "resnet18_aug_confusion_matrix_tta.csv"
    cm_img_path = evaluation_dir / "resnet18_aug_confusion_matrix_tta.png"

    save_metrics(metrics, metrics_path)
    save_classification_report(metrics["report_dict"], report_path)
    save_predictions(
        metrics["labels"],
        metrics["preds"],
        metrics["file_names"],
        metrics["class_names"],
        predictions_path,
    )
    save_confusion_matrix_csv(metrics["confusion_matrix"], cm_csv_path)
    plot_confusion_matrix(
        metrics["confusion_matrix"],
        CLASS_NAMES,
        cm_img_path,
        "ResNet18 + Augmentation TTA Confusion Matrix",
    )

    run = wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        group=WANDB_GROUP,
        job_type="evaluate",
        config={
            "model_name": "resnet18_aug",
            "checkpoint_path": str(checkpoint_path),
            "num_classes": len(CLASS_NAMES),
            "dropout": DROPOUT,
            "tta_shift": TTA_SHIFT,
        },
    )

    run.log({
        "test_loss": metrics["test_loss"],
        "test_accuracy": metrics["accuracy"],
        "test_macro_f1": metrics["macro_f1"],
        "confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=metrics["labels"],
            preds=metrics["preds"],
            class_names=CLASS_NAMES,
        ),
        "confusion_matrix_image": wandb.Image(str(cm_img_path)),
    })

    run.summary["metrics_path"] = str(metrics_path)
    run.summary["report_path"] = str(report_path)
    run.summary["predictions_path"] = str(predictions_path)
    run.summary["confusion_matrix_csv_path"] = str(cm_csv_path)
    run.summary["confusion_matrix_image_path"] = str(cm_img_path)
    run.finish()

    print("Evaluation finished.")
    print(f"Checkpoint: {checkpoint_path}")
    print(f"Test Loss: {metrics['test_loss']:.4f}")
    print(f"Test Accuracy: {metrics['accuracy']:.4f}")
    print(f"Test Macro-F1: {metrics['macro_f1']:.4f}")
    print()
    print(metrics["report_text"])
    print(f"Saved metrics to: {metrics_path}")
    print(f"Saved report to: {report_path}")
    print(f"Saved predictions to: {predictions_path}")
    print(f"Saved confusion matrix csv to: {cm_csv_path}")
    print(f"Saved confusion matrix image to: {cm_img_path}")


if __name__ == "__main__":
    main()


Writing evaluate_resnet_aug.py


In [28]:
!python evaluate_resnet_aug.py

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: tien-ngozack2004 (tien-ngozack2004-ho-chi-minh-city-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run xs53xtjo (0.5s)
wandb: ⣯ setting up run xs53xtjo (0.5s)
wandb: ⣟ setting up run xs53xtjo (0.5s)
wandb: ⡿ setting up run xs53xtjo (0.5s)
wandb: ⢿ setting up run xs53xtjo (0.5s)
wandb: ⣻ setting up run xs53xtjo (1.0s)
wandb: ⣽ setting up run xs53xtjo (1.0s)
wandb: ⣾ setting up run xs53xtjo (1.0s)
wandb: ⣷ setting up run xs53xtjo (1.0s)
wandb: ⣯ setting up run xs53xtjo (1.0s)
wandb: ⣟ setting up run xs53xtjo (1.5s)
wandb: ⡿ setting up run xs53xtjo (1.5s)
wandb: ⢿ setting up run xs53xtjo (1.5s)
wandb: ⣻ setting up run xs53xtjo (1.5s)
wandb: ⣽ setting up run xs5

In [29]:
%%writefile evaluate_ensemble.py
from pathlib import Path
import csv
import json

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import wandb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from config import CLASS_NAMES
from dataset_aug import UrbanSoundDataLoaderAug
from resnet import ResNet18


WANDB_PROJECT = "urban-sound-classification"
WANDB_RUN_NAME = "ensemble_resnet_reg_aug_tta_eval"
WANDB_GROUP = "ensemble"

RESNET_REG_TAG = "resnet18_regular_v3"
RESNET_REG_DEFAULT_BASE_CHANNELS = 32
RESNET_REG_DEFAULT_DROPOUT = 0.3

RESNET_AUG_BASE_CHANNELS = 64
RESNET_AUG_DROPOUT = 0.5

TTA_SHIFT = 4

ENSEMBLE_WEIGHT_REG = 0.5
ENSEMBLE_WEIGHT_AUG = 0.5


def forward_tta(model, features):
    outputs_1 = model(features)
    outputs_2 = model(torch.roll(features, shifts=TTA_SHIFT, dims=-1))
    outputs_3 = model(torch.roll(features, shifts=-TTA_SHIFT, dims=-1))
    return (outputs_1 + outputs_2 + outputs_3) / 3.0


def load_resnet_regular(checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
        base_channels = checkpoint.get("base_channels", RESNET_REG_DEFAULT_BASE_CHANNELS)
        dropout = checkpoint.get("dropout", RESNET_REG_DEFAULT_DROPOUT)
    else:
        state_dict = checkpoint
        base_channels = RESNET_REG_DEFAULT_BASE_CHANNELS
        dropout = RESNET_REG_DEFAULT_DROPOUT

    model = ResNet18(
        num_classes=len(CLASS_NAMES),
        base_channels=base_channels,
        dropout=dropout,
    ).to(device)
    model.load_state_dict(state_dict)
    return model, base_channels, dropout


def load_resnet_aug(checkpoint_path, device):
    state_dict = torch.load(checkpoint_path, map_location=device)
    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model = ResNet18(
        num_classes=len(CLASS_NAMES),
        base_channels=RESNET_AUG_BASE_CHANNELS,
        dropout=RESNET_AUG_DROPOUT,
    ).to(device)
    model.load_state_dict(state_dict)
    return model


def evaluate_ensemble(model_reg, model_aug, loader, criterion, device):
    model_reg.eval()
    model_aug.eval()

    running_loss = 0.0
    total = 0

    all_labels = []
    all_preds = []
    all_probs = []
    all_file_names = []
    all_class_names = []

    with torch.no_grad():
        for batch in loader:
            features = batch["feature"].to(device)
            labels = batch["label"].to(device)

            logits_reg = forward_tta(model_reg, features)
            logits_aug = forward_tta(model_aug, features)

            probs_reg = torch.softmax(logits_reg, dim=1)
            probs_aug = torch.softmax(logits_aug, dim=1)

            probs = (
                ENSEMBLE_WEIGHT_REG * probs_reg
                + ENSEMBLE_WEIGHT_AUG * probs_aug
            )

            loss = criterion(torch.log(probs + 1e-8), labels)
            preds = probs.argmax(dim=1)

            running_loss += loss.item() * features.size(0)
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
            all_file_names.extend(batch["file_name"])
            all_class_names.extend(batch["class_name"])

    test_loss = running_loss / total
    accuracy = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    cm = confusion_matrix(all_labels, all_preds)
    report_dict = classification_report(
        all_labels,
        all_preds,
        target_names=CLASS_NAMES,
        output_dict=True,
        digits=4,
        zero_division=0,
    )
    report_text = classification_report(
        all_labels,
        all_preds,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )

    return {
        "test_loss": test_loss,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "confusion_matrix": cm,
        "report_dict": report_dict,
        "report_text": report_text,
        "labels": all_labels,
        "preds": all_preds,
        "probs": all_probs,
        "file_names": all_file_names,
        "class_names": all_class_names,
    }


def save_metrics(metrics, save_path):
    data = {
        "test_loss": metrics["test_loss"],
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
    }

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)


def save_classification_report(report_dict, save_path):
    rows = []

    for label, values in report_dict.items():
        if isinstance(values, dict):
            row = {"label": label}
            row.update(values)
            rows.append(row)

    fieldnames = sorted({key for row in rows for key in row.keys()})

    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def save_predictions(labels, preds, probs, file_names, class_names, save_path):
    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "file_name",
            "true_class_name",
            "true_label",
            "pred_label",
            "pred_class_name",
            "pred_confidence",
        ])

        for file_name, true_class_name, true_label, pred_label, prob_vector in zip(
            file_names, class_names, labels, preds, probs
        ):
            pred_confidence = float(prob_vector[pred_label])
            writer.writerow([
                file_name,
                true_class_name,
                true_label,
                pred_label,
                CLASS_NAMES[pred_label],
                pred_confidence,
            ])


def save_confusion_matrix_csv(cm, save_path):
    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["true/pred"] + CLASS_NAMES)

        for class_name, row in zip(CLASS_NAMES, cm):
            writer.writerow([class_name] + row.tolist())


def plot_confusion_matrix(cm, class_names, save_path, title):
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm)

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.set_title(title)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")

    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    root_dir = Path(__file__).resolve().parent.parent
    checkpoint_reg_path = root_dir / "outputs" / "checkpoints" / f"{RESNET_REG_TAG}_best.pth"
    checkpoint_aug_path = root_dir / "outputs" / "checkpoints" / "resnet18_aug_best.pth"
    evaluation_dir = root_dir / "outputs" / "evaluation"
    evaluation_dir.mkdir(parents=True, exist_ok=True)

    if not checkpoint_reg_path.exists():
        raise FileNotFoundError(f"Không tìm thấy ResNet regular checkpoint: {checkpoint_reg_path}")
    if not checkpoint_aug_path.exists():
        raise FileNotFoundError(f"Không tìm thấy ResNet aug checkpoint: {checkpoint_aug_path}")

    data_module = UrbanSoundDataLoaderAug(num_workers=0)
    _, _, test_loader = data_module.get_dataloaders()

    model_reg, reg_base_channels, reg_dropout = load_resnet_regular(checkpoint_reg_path, device)
    model_aug = load_resnet_aug(checkpoint_aug_path, device)

    criterion = nn.NLLLoss()

    metrics = evaluate_ensemble(
        model_reg=model_reg,
        model_aug=model_aug,
        loader=test_loader,
        criterion=criterion,
        device=device,
    )

    metrics_path = evaluation_dir / "ensemble_test_metrics_tta.json"
    report_path = evaluation_dir / "ensemble_classification_report_tta.csv"
    predictions_path = evaluation_dir / "ensemble_test_predictions_tta.csv"
    cm_csv_path = evaluation_dir / "ensemble_confusion_matrix_tta.csv"
    cm_img_path = evaluation_dir / "ensemble_confusion_matrix_tta.png"

    save_metrics(metrics, metrics_path)
    save_classification_report(metrics["report_dict"], report_path)
    save_predictions(
        metrics["labels"],
        metrics["preds"],
        metrics["probs"],
        metrics["file_names"],
        metrics["class_names"],
        predictions_path,
    )
    save_confusion_matrix_csv(metrics["confusion_matrix"], cm_csv_path)
    plot_confusion_matrix(
        metrics["confusion_matrix"],
        CLASS_NAMES,
        cm_img_path,
        "Ensemble (ResNet18 + ResNet18-aug) TTA Confusion Matrix",
    )

    run = wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        group=WANDB_GROUP,
        job_type="evaluate",
        config={
            "model_name": "ensemble_resnet_reg_aug",
            "checkpoint_reg_path": str(checkpoint_reg_path),
            "checkpoint_aug_path": str(checkpoint_aug_path),
            "num_classes": len(CLASS_NAMES),
            "reg_base_channels": reg_base_channels,
            "reg_dropout": reg_dropout,
            "aug_base_channels": RESNET_AUG_BASE_CHANNELS,
            "aug_dropout": RESNET_AUG_DROPOUT,
            "ensemble_weight_reg": ENSEMBLE_WEIGHT_REG,
            "ensemble_weight_aug": ENSEMBLE_WEIGHT_AUG,
            "tta_shift": TTA_SHIFT,
        },
    )

    run.log({
        "test_loss": metrics["test_loss"],
        "test_accuracy": metrics["accuracy"],
        "test_macro_f1": metrics["macro_f1"],
        "confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=metrics["labels"],
            preds=metrics["preds"],
            class_names=CLASS_NAMES,
        ),
        "confusion_matrix_image": wandb.Image(str(cm_img_path)),
    })

    run.summary["metrics_path"] = str(metrics_path)
    run.summary["report_path"] = str(report_path)
    run.summary["predictions_path"] = str(predictions_path)
    run.summary["confusion_matrix_csv_path"] = str(cm_csv_path)
    run.summary["confusion_matrix_image_path"] = str(cm_img_path)
    run.finish()

    print("Ensemble evaluation finished.")
    print(f"ResNet regular checkpoint: {checkpoint_reg_path}")
    print(f"ResNet aug checkpoint:     {checkpoint_aug_path}")
    print(f"Ensemble weights: reg={ENSEMBLE_WEIGHT_REG}, aug={ENSEMBLE_WEIGHT_AUG}")
    print(f"Test Loss: {metrics['test_loss']:.4f}")
    print(f"Test Accuracy: {metrics['accuracy']:.4f}")
    print(f"Test Macro-F1: {metrics['macro_f1']:.4f}")
    print()
    print(metrics["report_text"])
    print(f"Saved metrics to: {metrics_path}")
    print(f"Saved report to: {report_path}")
    print(f"Saved predictions to: {predictions_path}")
    print(f"Saved confusion matrix csv to: {cm_csv_path}")
    print(f"Saved confusion matrix image to: {cm_img_path}")


if __name__ == "__main__":
    main()


Overwriting evaluate_ensemble.py


In [30]:
!python evaluate_ensemble.py

Streaming output truncated to the last 5000 lines.
wandb: ⡿ uploading summary (1.2m)
wandb:   ERROR retrying HTTP 500: context deadline exceeded
wandb: ⢿ uploading artifact run-p0vednnh-confusion_matrix_table (1.2m)
wandb: ⢿ uploading summary (1.2m)
wandb:   ERROR retrying HTTP 500: context deadline exceeded
wandb: ⣻ uploading artifact run-p0vednnh-confusion_matrix_table (1.2m)
wandb: ⣻ uploading summary (1.2m)
wandb:   ERROR retrying HTTP 500: context deadline exceeded
wandb: ⣽ uploading artifact run-p0vednnh-confusion_matrix_table (1.2m)
wandb: ⣽ uploading summary (1.2m)
wandb:   ERROR retrying HTTP 500: context deadline exceeded
wandb: ⣾ uploading artifact run-p0vednnh-confusion_matrix_table (1.2m)
wandb: ⣾ uploading summary (1.2m)
wandb:   ERROR retrying HTTP 500: context deadline exceeded
wandb: ⣷ uploading artifact run-p0vednnh-confusion_matrix_table (1.2m)
wandb: ⣷ uploading summary (1.2m)
wandb:   ERROR retrying HTTP 500: context deadline exceeded
wandb: ⣯ uploading artifact ru